In [18]:
import numpy as np
import matplotlib.pyplot as plt
import json
import re

import os

In [19]:
def read_file_to_json(fnam):

    with open(fnam) as f:
        data = {}
        text = ""
        key = None
        for i,l in enumerate(f):
            if i == 0:
                name = l.rstrip().replace("# ","")
                data["Name"] = name
                continue

            if "##" in l:
                if key != None:
                    data[key] = text
                key = l.replace("## ","").rstrip()
                text = ""
            else:
                text += l.rstrip()
        data[key] = text 
    return data
fold1 = "curvatures/"
fold2 = "global_charges/"
fold3 = "global_charges/"
folds = [fold1,fold2,fold3]
data = {}

for fold in folds:
    for f in os.listdir(fold):
        if f[0] == ".":
            continue
        
        data[f] = read_file_to_json(fold + f)
        data[f]['folder'] = fold
        data[f]['category'] = fold.replace("/","")
        data[f]['writefolder'] = "../colls/_{}".format(fold)

In [20]:
with open("references.json", 'r') as f:
    citations_json = json.load(f)
citations_json

{'ozawa2019topological': {'url': 'https://journals.aps.org/rmp/abstract/10.1103/RevModPhys.91.015006'}}

In [23]:
def extract_connected_quantities_from_text(text):
    connections = re.findall(r'\[([A_Za-z0-9_]+).md\]',text)
    return [c + ".md" for c in connections]
extract_connected_quantities_from_text(data['berry-connection.md']['Text'])

['pbphase.md', 'berry_curvature.md']

In [31]:
def_pre = """---
layout: post
title: [title]
category: [category]
---"""

def generate_markdown(data_full, curr, preamble = def_pre, citations_json = citations_json):
    data = data_full[curr]
    outf = def_pre
    outf = outf.replace("[title]",data["Name"])
    outf = outf.replace("[category]",data["category"])

    outf += "\n"
    text = data["Text"]
    text = text.replace("[equation]","\n\n$${}$$\n".format(data['Equation']))
    outf += text

    connected_quantities = extract_connected_quantities_from_text(data["Text"])

    if len(connected_quantities) > 0:
        outf += """
        
        ## Connected quantities
    
        | Quantity | connection |
        | --- | --- |
        """.replace("    ","")
        quants = connected_quantities
        for q in quants:
            links = "[{}]".format(data_full[q]['Name']) + "{{ site.baseurl }}{% link " + "../_{}/{}".format(data_full[q]["category"],q) + " %}"
            outf += "| [{}] | ${}$ |\n".format(q,data_full[q]["Equation"])
            outf = outf.replace("[{}]".format(q),links)

    outf += "\n"
    if data["Alternative symbols"] != "":
        outf += """
        ### Alternative symbols
        
        | Symbol | Work |
        | --- | --- |
        """.replace("    ","")
        alts = data['Alternative symbols'].split(";")
        for a in alts:
            symb, ref = a[1:-1].split(",")
            outf += "| {} | [{}] |\n".format(symb.rstrip(),ref.replace(" ", ""))

    if data["Citations"] != "":
        outf += """
        
        ### Citations
        """.replace("    ","")
        citations = data['Citations'].split(",")
        first_citation = np.array([outf.find(cit) for cit in citations])
        idx = np.argsort(first_citation)

        for i in idx:
            ii = i+1
            outf = outf.replace("[{}]".format(citations[i]),"[{}]".format(ii))
            cit = citations[i]
            outf += "[[{}] {}]({})".format(ii,citations_json[cit]['url'],citations_json[cit]['url'])
        
    return outf    

        
pbphase_md = generate_markdown(data, "berry-connection.md")
print(pbphase_md)

---
layout: post
title: Berry Connection
category: curvatures
---
Berry connection is useful [1]

$$\mathcal{A}_n(\mathbf{k}) = i\langle u_n(\mathbf{k})|\nabla_\mathbf{k}|u_n(\mathbf{k})\rangle$$
When integrated over a loop, it gives the [Pancharatnam-Berry phase]{{ site.baseurl }}{% link ../_curvatures/pbphase.md %}.It's curl gives the [Berry Curvature]{{ site.baseurl }}{% link ../_curvatures/berry_curvature.md %}.

## Connected quantities

| Quantity | connection |
| --- | --- |
| [Pancharatnam-Berry phase]{{ site.baseurl }}{% link ../_curvatures/pbphase.md %} | $\gamma = \oint_\mathcal{C}d\mathbf{k}\cdot\mathcal{A}(\mathbf{k})$ |
| [Berry Curvature]{{ site.baseurl }}{% link ../_curvatures/berry_curvature.md %} | $\mathcal{B}(\mathbf{k}) = \nabla \times \mathcal{A}(\mathbf{k})$ |


### Alternative symbols

| Symbol | Work |
| --- | --- |
| $\Omega(\mathbf{k})$ | [1] |


### Citations
[[1] https://journals.aps.org/rmp/abstract/10.1103/RevModPhys.91.015006](https://journals.aps.org/rmp

In [18]:
def write_pages(data,citations_json = citations_json):
    for d in data:
        outf = data[d]["writefolder"] + d
        
        file_md = generate_markdown(data,d,citations_json=citations_json)

        print("writing to", outf)
        with open(outf,"w") as f:
            f.write(file_md)
        
write_pages(data)

writing to ../colls/_curvatures/berry_curvature.md
writing to ../colls/_curvatures/pbphase.md


In [33]:
A = np.array([4,2,1,5])
idx = np.argsort(A)

In [34]:
for i in idx:
    print(A[i])

1
2
4
5


In [88]:
text = """
The handedness weighed sub-charges are phase-singularities in the left- and right-handed electric field components.
They are defined as

They are connected to the [bic_charge.md] as
"""
text

'\nThe handedness weighed sub-charges are phase-singularities in the left- and right-handed electric field components.\nThey are defined as\n\nThey are connected to the [bic_charge.md] as\n'

In [91]:
def extract_connected_quantities_from_text(text):
    connections = re.findall(r'\[([A_Za-z0-9_,]+).md\]',text)
    connections = list(set(connections))

    conns = {}
    
    for c in connections:
        cs = c.split(",")
        if len(cs) == 1:
            idx = 1
            conn = cs[0]
        else:
            idx, conn = c.split(",")
            idx = int(idx)
        conns[conn + ".md"] = "equation{}".format(idx)
    return conns

In [96]:
quants = extract_connected_quantities_from_text(text)
for q in quants:
    print(q, quants[q]):w

bic_charge.md equation1


In [53]:
eq = "$q = \\frac{q_- - q_+}{2}$\n$q1 = \\frac{q_- - q_+}{2}$\n$q2 = \\frac{q_- - q_+}{2}$"
re.findall(r"\$(.*?)\$", eq)

['q = \\frac{q_- - q_+}{2}',
 'q1 = \\frac{q_- - q_+}{2}',
 'q2 = \\frac{q_- - q_+}{2}']

In [95]:

def extract_equations(equations):
    eqs = re.findall(r'\$(.*?)\$', equations)
    return {"equation{}".format(i + 1) : eqs[i] for i in range(len(eqs))}


3

In [100]:
A = [1,2,3]
B = [3,5,1]
#A.append(B)
A += B
A

[1, 2, 3, 3, 5, 1]

In [101]:
a = "asd"
type(a) == str

True

In [125]:
class equation:

    def __init__(self, eq, page):
        self.eq = eq
        self.pages = [page]

    def __eq__(self, other):
        if type(other) is str:
            print("comparing with string")
            
            return self.eq == other

        if type(other) is type(self):
            return self.eq == other.eq

        print("Types not matching")

    def __add__(self, other):
        self.pages += other.pages
        return self


In [126]:
eq1 = equation("a + b = c","")
eq2 = equation("a + b = c","")

In [127]:
eq1 == eq2

True

In [128]:
CA = np.array([
    [1,1,1,1],
    [1,-1,1,-1],
    [1,1,-1,-1],
    [1,-1,-1,1]
])
np.linalg.eig(CA)

EigResult(eigenvalues=array([ 2., -2., -2.,  2.]), eigenvectors=array([[ 0.8660254 ,  0.5       ,  0.09635219,  0.1828397 ],
       [ 0.28867513, -0.5       , -0.79020557,  0.45999255],
       [ 0.28867513, -0.5       ,  0.59750119,  0.45999255],
       [ 0.28867513, -0.5       , -0.09635219, -0.73714541]]))

In [129]:
import sympy as sp
from sympy import Matrix

In [130]:
CA = Matrix([
    [1,1,1,1],
    [1,-1,1,-1],
    [1,1,-1,-1],
    [1,-1,-1,1]
])
CA

Matrix([
[1,  1,  1,  1],
[1, -1,  1, -1],
[1,  1, -1, -1],
[1, -1, -1,  1]])

In [133]:
CA.diagonalize()

(Matrix([
 [ 0, -1, 2, 1],
 [-1,  2, 1, 0],
 [ 1,  0, 1, 0],
 [ 0,  1, 0, 1]]),
 Matrix([
 [-2,  0, 0, 0],
 [ 0, -2, 0, 0],
 [ 0,  0, 2, 0],
 [ 0,  0, 0, 2]]))